# Identificação do projeto

**Título do projeto:** Dengue no Brasil em 2026: pico de casos, gravidade e perfil dos pacientes

**Integrantes do grupo:** Bruna da Silva Carnelossi, João Pedro de Souza Quintiliano da Silva, João Victor Carvalho de Freitas, Matheus Henrique Heinzen

**Turma:** Sistemas de Informação • 6º período • Turma A GCL1 • Profª Marina de Lara

**Descrição breve do problema ou contexto:** a dengue é a arbovirose de maior impacto no Brasil e todo caso suspeito é de notificação compulsória ao SINAN. O projeto usa as notificações de 2026 para entender quando os casos se concentraram, onde e quais perfis de paciente evoluem para as formas graves da doença.

Repositório: https://github.com/JoaoVictorCarvalho07/api-dados-abertos-saude

---

# RA1: Data Ingestion, Cleaning and Initial Analysis

**Fontes de dados → Ingestão → Compreensão → Avaliação da qualidade → Limpeza e preparação → Dados preparados**

O fluxo segue a lógica **Inspecionar → Decidir → Transformar → Validar**: nenhuma alteração é feita antes de o problema ser medido, e nenhuma alteração é dada por concluída antes de ser verificada.

> `df_original` guarda exatamente o que veio do arquivo oficial e nunca é alterado. Toda a preparação acontece numa cópia, `df_limpo`.

## Preparação do ambiente

Bibliotecas, parâmetros do projeto e limites usados nas regras de qualidade. Os limites ficam aqui, num só lugar: se for necessário mudar um critério (por exemplo, o atraso considerado extremo), todas as verificações acompanham.

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 90)

RAIZ = Path.cwd().parent
URL_DADOS = "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/csv/DENGBR26.csv.zip"
ARQUIVO_BRUTO = RAIZ / "DENGBR26.csv.zip"
ARQUIVO_PREPARADO = RAIZ / "dados" / "preparados" / "dengue_brasil_2026.parquet"

INICIO_PERIODO = pd.Timestamp("2025-12-01")
FIM_PERIODO = pd.Timestamp("2026-10-10")
ANO_REFERENCIA = 2026
IDADE_MAXIMA = 120
ATRASO_EXTREMO_DIAS = 60


def milhar(numero):
    return f"{numero:,}".replace(",", ".")


def percentual(parte, total):
    return f"{parte / total:.2%}".replace(".", ",")


print("✓ Ambiente preparado.")
print(f"pandas {pd.__version__}")
print(f"Período válido para datas: {INICIO_PERIODO.date()} a {FIM_PERIODO.date()}")

## 1. Contexto e objetivo

Apresente brevemente o problema investigado e o que o grupo pretende descobrir com os dados.

**Resposta do grupo:**

**Contexto e problema.** A dengue é uma doença de notificação compulsória: todo caso suspeito deve ser registrado no SINAN (Sistema de Informação de Agravos de Notificação). Em anos epidêmicos, os casos se concentram em poucas semanas e pressionam a rede de saúde, e uma parte dos pacientes evolui para dengue com sinais de alarme ou dengue grave, que exigem acompanhamento e internação. Entender **quando** os casos se concentram, **onde** e **quem** tem mais chance de agravar ajuda gestores a planejar leitos, campanhas e vigilância.

**Objetivo.** Preparar uma base confiável das notificações de dengue de 2026 que permita investigar a evolução dos casos no tempo, a distribuição entre os estados e a relação entre gravidade, sexo, idade e comorbidades.

**Recorte:**
-  **Período:**  Notificações de 2026 (o arquivo cobre de 04/01 a 04/10/2026) 
-  **Abrangência:**  Brasil, todas as UFs e regiões 
-  **Unidade de análise:** Notificação de caso de dengue 
-  **Limite:** Somente dengue; chikungunya e zika ficam fora 

**Perguntas analíticas.** Neste RA as perguntas apenas orientam a seleção e a preparação dos dados; as respostas serão desenvolvidas nos próximos RAs.

| ID | Pergunta a investigar | Dados necessários |
|---|---|---|
| P1 | Em que mês de 2026 os casos de dengue atingiram o pico no Brasil e qual estado concentrou mais casos? | `DT_SIN_PRI` (data de início dos sintomas), `SG_UF` (UF de residência) |
| P2 | A proporção de casos graves (classificação final "dengue com sinais de alarme" ou "dengue grave") é maior entre homens ou entre mulheres? | `CS_SEXO`, `CLASSI_FIN` |
| P3 | A proporção de casos graves varia por faixa etária e pela presença de comorbidades (diabetes e hipertensão)? | `NU_IDADE_N`, `ANO_NASC`, `CLASSI_FIN`, `DIABETES`, `HIPERTENSA` |

Duas definições fixadas pelo grupo para que as perguntas sejam mensuráveis:

- **Caso grave** = `CLASSI_FIN` 11 (dengue com sinais de alarme) ou 12 (dengue grave). A proporção é calculada sobre os casos classificados como dengue (10, 11 e 12); casos inconclusivos ou ainda em investigação ficam fora do denominador.
- **Estado mais afetado** = UF de **residência**, e não de notificação: a residência indica onde a doença circula, enquanto a notificação indica onde a pessoa foi atendida. As duas ficam disponíveis na base preparada.

**Viabilidade inicial.** A base traz, para cada notificação, a data de início dos sintomas, a UF e o município de residência, sexo, idade, comorbidades e a classificação final do caso, o suficiente para as três perguntas. As lacunas conhecidas são: o ano ainda não terminou (os últimos meses estão incompletos e parte dos casos segue em investigação); não há população por UF, então a base não permite calcular incidência por habitante sem uma segunda fonte; e não existe identificador de paciente.

## 2. Fonte e ingestão dos dados

- Identifique a fonte dos dados, incluindo link ou referência de acesso.
- Descreva brevemente o dataset e o que seus registros representam.
- Explique como os dados foram carregados e implemente a ingestão.

**Resposta do grupo:**

| Item | Descrição |
|---|---|
| Conjunto de dados | Sinan/Dengue, do Portal de Dados Abertos do SUS, Ministério da Saúde |
| Página oficial | https://dadosabertos.saude.gov.br/dataset/arboviroses-dengue |
| Arquivo | `DENGBR26.csv.zip`, recurso "Dengue - 2026", atualizado em 10/10/2026 |
| Download direto | https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/csv/DENGBR26.csv.zip |
| Formato | CSV compactado em zip, separador vírgula, codificação UTF-8 |
| Volume | ~470 mil notificações x 121 colunas |
| O que cada linha representa | Uma notificação de caso de dengue registrada no SINAN |
| Condições de uso | Dados abertos e públicos, anonimizados (sem nome nem identificador do paciente) |
| Documentação | Dicionário de dados oficial `dic_dados_dengue.pdf`, na raiz do repositório |
| Relação com o problema | Traz data dos sintomas e UF de residência (P1), sexo e classificação final (P2), idade e comorbidades (P3) |

O mesmo portal oferece uma API (`apidadosabertos.saude.gov.br/arboviroses/dengue`), mas ela só filtra por município e devolve no máximo 1.000 registros por requisição; o arquivo anual traz o país inteiro de uma vez e é a fonte usada pelo projeto.

**Como a ingestão funciona:**

1. Se o zip não estiver na raiz do repositório, o código o baixa da URL oficial; assim, quem clonar o projeto reproduz a coleta sem passo manual.
2. O arquivo é lido direto do zip, sem descompactar no disco.
3. **Todas as colunas são lidas como texto.** Se o pandas inferisse os tipos sozinho, códigos como municípios e classificações virariam números e datas ficariam como objetos genéricos. Ler como texto preserva o dado exatamente como veio; a conversão de tipos é uma decisão de limpeza, feita na seção 4 com regra explícita.
4. A verificação do carregamento compara o número de linhas do arquivo com o número de linhas que o pandas carregou.

In [ ]:
if not ARQUIVO_BRUTO.exists():
    urllib.request.urlretrieve(URL_DADOS, ARQUIVO_BRUTO)

with zipfile.ZipFile(ARQUIVO_BRUTO) as pacote:
    conteudo = pacote.infolist()[0]
    with pacote.open(conteudo) as arquivo:
        cabecalho = arquivo.readline().decode("utf-8")
        linhas_no_arquivo = sum(1 for _ in arquivo)

separador = max([",", ";", "|"], key=cabecalho.count)

print("ARQUIVO BRUTO")
print("─" * 50)
print(f"Arquivo:                 {ARQUIVO_BRUTO.name}")
print(f"Conteúdo do zip:         {conteudo.filename}")
print(f"Tamanho compactado:      {ARQUIVO_BRUTO.stat().st_size / 1e6:.1f} MB")
print(f"Tamanho descompactado:   {conteudo.file_size / 1e6:.1f} MB")
print(f"Separador encontrado:    '{separador}'")
print(f"Colunas no cabeçalho:    {cabecalho.count(separador) + 1}")
print(f"Linhas de dados:         {milhar(linhas_no_arquivo)}")

In [ ]:
df_original = pd.read_csv(
    ARQUIVO_BRUTO,
    sep=separador,
    encoding="utf-8",
    dtype="string[pyarrow]",
)

print("CARREGAMENTO")
print("─" * 50)
print(f"Shape:                   {milhar(df_original.shape[0])} linhas × {df_original.shape[1]} colunas")
print(f"Memória em uso:          {df_original.memory_usage(deep=True).sum() / 1e6:.0f} MB")
print(f"Linhas no arquivo:       {milhar(linhas_no_arquivo)}")
print(f"Linhas carregadas:       {milhar(len(df_original))}")

assert len(df_original) == linhas_no_arquivo, "O pandas não carregou todas as linhas do arquivo."
print("✓ Todas as linhas do arquivo foram carregadas.")

print("\nPrimeiras linhas (primeiras 14 colunas):")
display(df_original.iloc[:5, :14])

## 3. Exploração inicial dos dados

Investigue a quantidade de registros e colunas, os tipos de dados, os valores ausentes e outras características relevantes da base.

### 3.1 Conhecer os dados

**Unidade de observação:** uma linha representa **uma notificação de caso de dengue**.

**Identificação dos registros:** a base pública não traz o número da notificação nem qualquer identificador do paciente. Não existe, portanto, uma chave que garanta unicidade. Duas linhas idênticas em todas as 121 colunas são tratadas como o mesmo registro repetido; linhas muito parecidas só podem ser apontadas como *possíveis* duplicatas.

Antes de procurar problemas, registramos o significado e o domínio das colunas que as perguntas usam. Os códigos seguem o dicionário de dados do SINAN (`dic_dados_dengue.pdf`).

In [ ]:
UF_POR_CODIGO = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA", "16": "AP", "17": "TO",
    "21": "MA", "22": "PI", "23": "CE", "24": "RN", "25": "PB", "26": "PE", "27": "AL", "28": "SE", "29": "BA",
    "31": "MG", "32": "ES", "33": "RJ", "35": "SP",
    "41": "PR", "42": "SC", "43": "RS",
    "50": "MS", "51": "MT", "52": "GO", "53": "DF",
}
REGIAO_POR_PRIMEIRO_DIGITO = {"1": "Norte", "2": "Nordeste", "3": "Sudeste", "4": "Sul", "5": "Centro-Oeste"}

CLASSIFICACAO_FINAL = {
    "5": "Descartado",
    "8": "Inconclusivo",
    "10": "Dengue",
    "11": "Dengue com sinais de alarme",
    "12": "Dengue grave",
    "13": "Chikungunya",
}
CRITERIO_CONFIRMACAO = {"1": "Laboratorial", "2": "Clínico-epidemiológico", "3": "Em investigação"}
EVOLUCAO_INFORMADA = {"1": "Cura", "2": "Óbito pelo agravo", "3": "Óbito por outras causas", "4": "Óbito em investigação"}
SEXO_INFORMADO = {"M": "Masculino", "F": "Feminino"}
SIM_NAO = {"1": True, "2": False}
FATOR_IDADE_EM_ANOS = {"1": 1 / (24 * 365), "2": 1 / 365, "3": 1 / 12, "4": 1}

dicionario = pd.DataFrame(
    [
        ("DT_NOTIFIC", "Data da notificação", "AAAA-MM-DD", "datas de 2026", "P1"),
        ("DT_SIN_PRI", "Data de início dos sintomas", "AAAA-MM-DD", "não pode ser posterior à notificação", "P1"),
        ("SEM_PRI", "Semana epidemiológica dos sintomas", "AAAASS", "semanas de 2025-2026", "P1"),
        ("SG_UF_NOT / ID_MUNICIP", "UF e município de notificação", "código IBGE", "27 UFs; município com 6 dígitos", "P1"),
        ("SG_UF / ID_MN_RESI", "UF e município de residência", "código IBGE", "27 UFs; município com 6 dígitos", "P1"),
        ("CS_SEXO", "Sexo", "letra", "M, F, I = ignorado", "P2"),
        ("NU_IDADE_N", "Idade codificada", "4 dígitos", "1º dígito = unidade (1 hora, 2 dia, 3 mês, 4 ano); demais = valor", "P3"),
        ("ANO_NASC", "Ano de nascimento", "AAAA", "alternativa para a idade", "P3"),
        ("CLASSI_FIN", "Classificação final", "código", "5, 8, 10, 11, 12, 13; vazio = em investigação", "P2, P3"),
        ("CRITERIO", "Critério de confirmação", "código", "1 laboratorial, 2 clínico-epidemiológico, 3 em investigação", "apoio"),
        ("HOSPITALIZ", "Houve hospitalização", "código", "1 sim, 2 não, 9 ignorado", "apoio"),
        ("EVOLUCAO", "Evolução do caso", "código", "1 cura, 2 óbito pelo agravo, 3 outras causas, 4 em investigação, 9 ignorado", "apoio"),
        ("DIABETES / HIPERTENSA", "Comorbidades", "código", "1 sim, 2 não, 9 ignorado", "P3"),
    ],
    columns=["coluna", "significado", "formato", "domínio esperado", "uso"],
)
display(dicionario.style.hide(axis="index"))

### 3.2 Visão geral da base

Nesta etapa **ainda não alteramos nada**. Como a leitura foi feita como texto, todas as colunas aparecem com o mesmo tipo: isso é intencional, e a conversão será uma decisão registrada na seção 4.

In [ ]:
print("VISÃO GERAL DA BASE ORIGINAL")
print("─" * 50)
print(f"Linhas:   {milhar(df_original.shape[0])}")
print(f"Colunas:  {df_original.shape[1]}")
print(f"Células:  {milhar(df_original.size)}")

tipos = df_original.dtypes.astype(str).value_counts().rename_axis("tipo").reset_index(name="colunas")
print("\nTipos das colunas após a leitura:")
display(tipos.style.hide(axis="index"))

valores_unicos = df_original.nunique()
constantes = valores_unicos[(valores_unicos == 1) & df_original.notna().all()].index.tolist()
vazias = df_original.columns[df_original.isna().all()].tolist()
print(f"Colunas 100% vazias:         {len(vazias)}")
print(f"Colunas com um único valor:  {len(constantes)} → {', '.join(constantes)}")

### 3.3 Valores ausentes

Quantidade e percentual de ausências, primeiro para a base inteira (por faixas) e depois para as colunas usadas pelas perguntas.

In [ ]:
percentual_ausente = df_original.isna().mean().mul(100)

faixas = pd.cut(
    percentual_ausente,
    bins=[-0.1, 0, 10, 50, 90, 100],
    labels=["0%", "até 10%", "10% a 50%", "50% a 90%", "acima de 90%"],
)
print("COLUNAS POR FAIXA DE AUSÊNCIA (121 colunas)")
display(faixas.value_counts().sort_index().rename_axis("ausência").reset_index(name="colunas").style.hide(axis="index"))

COLUNAS_DAS_PERGUNTAS = [
    "DT_NOTIFIC", "DT_SIN_PRI", "SEM_PRI", "SG_UF_NOT", "ID_MUNICIP", "SG_UF", "ID_MN_RESI",
    "CS_SEXO", "NU_IDADE_N", "ANO_NASC", "CLASSI_FIN", "CRITERIO", "HOSPITALIZ", "EVOLUCAO",
    "DIABETES", "HIPERTENSA",
]
ausencias = pd.DataFrame({
    "quantidade ausente": df_original[COLUNAS_DAS_PERGUNTAS].isna().sum(),
    "percentual ausente (%)": percentual_ausente[COLUNAS_DAS_PERGUNTAS],
}).sort_values("percentual ausente (%)", ascending=False)

print("\nAUSÊNCIAS NAS COLUNAS USADAS PELAS PERGUNTAS")
display(ausencias.style.format({"quantidade ausente": "{:,.0f}", "percentual ausente (%)": "{:.2f}%"}))

com_ausencia = ausencias[ausencias["quantidade ausente"] > 0]
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.barh(com_ausencia.index[::-1], com_ausencia["percentual ausente (%)"][::-1], color="#4C78A8")
ax.set_title("Percentual de valores ausentes nas colunas das perguntas")
ax.set_xlabel("Percentual ausente (%)")
for posicao, valor in enumerate(com_ausencia["percentual ausente (%)"][::-1]):
    ax.text(valor + 0.3, posicao, f"{valor:.1f}%", va="center", fontsize=9)
ax.set_xlim(0, max(com_ausencia["percentual ausente (%)"].max() * 1.2, 5))
plt.tight_layout()
plt.show()

### 3.4 Registros duplicados

Sem identificador, a verificação é feita em dois níveis:

- **duplicata exata**: a linha se repete em todas as 121 colunas, ou seja, é o mesmo registro gravado duas vezes;
- **possível duplicata**: mesma unidade de saúde, município, datas, idade, sexo e residência, mas com alguma diferença nas demais colunas; pode ser a mesma pessoa notificada duas vezes ou duas pessoas parecidas.

In [ ]:
CHAVE_POSSIVEL_DUPLICATA = [
    "ID_MUNICIP", "ID_UNIDADE", "DT_NOTIFIC", "DT_SIN_PRI",
    "ANO_NASC", "NU_IDADE_N", "CS_SEXO", "ID_MN_RESI",
]

duplicatas_exatas = int(df_original.duplicated().sum())
mesma_chave = int(df_original.duplicated(CHAVE_POSSIVEL_DUPLICATA).sum())
possiveis_duplicatas = mesma_chave - duplicatas_exatas

print("DUPLICATAS")
print("─" * 50)
print(f"Duplicatas exatas (121 colunas iguais):   {milhar(duplicatas_exatas)} ({percentual(duplicatas_exatas, len(df_original))})")
print(f"Possíveis duplicatas (mesma chave):        {milhar(possiveis_duplicatas)} ({percentual(possiveis_duplicatas, len(df_original))})")

print("\nExemplo de duplicata exata:")
exemplo = df_original[df_original.duplicated(keep=False)].sort_values(CHAVE_POSSIVEL_DUPLICATA).head(2)
display(exemplo[["DT_NOTIFIC", "SG_UF_NOT", "ID_MUNICIP", "DT_SIN_PRI", "NU_IDADE_N", "CS_SEXO", "CLASSI_FIN", "DT_ENCERRA"]])
print("⚠ Sem identificador de paciente, as possíveis duplicatas não podem ser confirmadas.")

### 3.5 Datas: formato, período e ordem

Três verificações: se a data segue o formato `AAAA-MM-DD`, se está dentro de um período plausível para notificações de 2026 e se a ordem entre as datas faz sentido (os sintomas começam antes da notificação; digitação e encerramento acontecem depois dela).

In [ ]:
COLUNAS_DATA = [coluna for coluna in df_original.columns if coluna.startswith("DT_")]
datas = df_original[COLUNAS_DATA].apply(pd.to_datetime, format="%Y-%m-%d", errors="coerce")

fora_do_periodo = (datas < INICIO_PERIODO) | (datas > FIM_PERIODO)
resumo_datas = pd.DataFrame({
    "preenchidas": df_original[COLUNAS_DATA].notna().sum(),
    "formato inválido": (df_original[COLUNAS_DATA].notna() & datas.isna()).sum(),
    "fora do período": fora_do_periodo.sum(),
    "menor data": datas.min().dt.date,
    "maior data": datas.max().dt.date,
})
display(resumo_datas.style.format({"preenchidas": "{:,.0f}"}))

datas_formato_invalido = int(resumo_datas["formato inválido"].sum())
registros_com_data_fora = int(fora_do_periodo.any(axis=1).sum())
sintoma_depois_notificacao = int((datas["DT_SIN_PRI"] > datas["DT_NOTIFIC"]).sum())
digitacao_antes_notificacao = int((datas["DT_DIGITA"] < datas["DT_NOTIFIC"]).sum())
encerramento_antes_notificacao = int((datas["DT_ENCERRA"] < datas["DT_NOTIFIC"]).sum())

print("ORDEM ENTRE AS DATAS")
print("─" * 50)
print(f"Registros com alguma data fora do período:    {milhar(registros_com_data_fora)}")
print(f"Sintomas depois da notificação:               {milhar(sintoma_depois_notificacao)}")
print(f"Digitação antes da notificação:               {milhar(digitacao_antes_notificacao)}")
print(f"Encerramento antes da notificação:            {milhar(encerramento_antes_notificacao)}")

### 3.6 Categorias e códigos

Verificamos se os valores de cada coluna codificada pertencem ao domínio do dicionário. Um código fora do domínio não tem significado e precisa de decisão antes de qualquer análise.

In [ ]:
classificacao = df_original["CLASSI_FIN"]
tabela_classificacao = classificacao.fillna("<vazio>").value_counts().rename_axis("código").reset_index(name="notificações")
tabela_classificacao["significado"] = (
    tabela_classificacao["código"]
    .map({**CLASSIFICACAO_FINAL, "<vazio>": "Vazio (caso ainda em investigação)"})
    .fillna("⚠ fora do dicionário")
)
print("CLASSIFICAÇÃO FINAL (CLASSI_FIN)")
display(tabela_classificacao.style.format({"notificações": "{:,.0f}"}).hide(axis="index"))

classificacao_fora_dominio = int((classificacao.notna() & ~classificacao.isin(list(CLASSIFICACAO_FINAL))).sum())
classificacao_vazia = int(classificacao.isna().sum())

uf_de_notificacao = df_original["SG_UF_NOT"].map(UF_POR_CODIGO)
codigo_zero = (classificacao == "0").fillna(False).astype(bool)
zero_por_uf = codigo_zero.groupby(uf_de_notificacao).agg(["sum", "mean"])
zero_por_uf = zero_por_uf[zero_por_uf["sum"] > 0].rename(columns={"sum": "notificações com '0'", "mean": "% das notificações da UF"})
print("\nOnde aparece o código '0':")
display(zero_por_uf.style.format({"notificações com '0'": "{:,.0f}", "% das notificações da UF": "{:.1%}"}))

In [ ]:
def contagem(coluna, dominio):
    valores = df_original[coluna].fillna("<vazio>").value_counts()
    return pd.DataFrame({
        "coluna": coluna,
        "valor": valores.index,
        "registros": valores.values,
        "no domínio": [valor in dominio for valor in valores.index],
    })

dominios = {
    "CS_SEXO": {"M", "F", "I"},
    "HOSPITALIZ": {"1", "2", "9"},
    "DIABETES": {"1", "2", "9"},
    "HIPERTENSA": {"1", "2", "9"},
    "EVOLUCAO": {"1", "2", "3", "4", "9"},
    "CRITERIO": {"1", "2", "3"},
}
categorias = pd.concat([contagem(coluna, dominio) for coluna, dominio in dominios.items()], ignore_index=True)
print("VALORES DAS DEMAIS COLUNAS CODIFICADAS")
display(categorias.style.format({"registros": "{:,.0f}"}).hide(axis="index"))

sexo_sem_informacao = int((~df_original["CS_SEXO"].isin(["M", "F"])).sum())
diabetes_sem_informacao = int((~df_original["DIABETES"].isin(["1", "2"])).sum())
hipertensao_sem_informacao = int((~df_original["HIPERTENSA"].isin(["1", "2"])).sum())
hospitalizado_sem_data = int(((df_original["HOSPITALIZ"] == "1") & df_original["DT_INTERNA"].isna()).sum())
uf_residencia_invalida = int((~df_original["SG_UF"].isin(list(UF_POR_CODIGO))).sum())

### 3.7 Idade

`NU_IDADE_N` mistura **unidade** e **valor** num único número: o primeiro dígito é a unidade (1 = horas, 2 = dias, 3 = meses, 4 = anos) e os demais, o valor. Assim, `4026` significa 26 anos e `3008` significa 8 meses. Usar a coluna como número direto faria um bebê de 8 meses aparecer com "3.008 anos".

In [ ]:
codigo_idade = df_original["NU_IDADE_N"].astype("string")
unidade_idade = codigo_idade.str[0]
valor_idade = pd.to_numeric(codigo_idade.str[1:], errors="coerce")
idade_declarada = valor_idade * unidade_idade.map(FATOR_IDADE_EM_ANOS).astype("float")
idade_pelo_nascimento = ANO_REFERENCIA - pd.to_numeric(df_original["ANO_NASC"], errors="coerce")

unidades = unidade_idade.map({"1": "1: horas", "2": "2: dias", "3": "3: meses", "4": "4: anos"}).fillna("inválida ou vazia")
print("UNIDADE DA IDADE (1º dígito de NU_IDADE_N)")
display(unidades.value_counts().rename_axis("unidade").reset_index(name="registros").style.format({"registros": "{:,.0f}"}).hide(axis="index"))

idade_invalida = int((idade_declarada.isna()).sum())
idade_acima_do_limite = int((idade_declarada > IDADE_MAXIMA).sum())
idade_divergente = int(((unidade_idade == "4") & ((idade_declarada - idade_pelo_nascimento).abs() > 1)).sum())

print("VERIFICAÇÕES DE IDADE")
print("─" * 50)
print(f"Idade vazia ou com unidade inválida:         {idade_invalida}")
print(f"Idade acima de {IDADE_MAXIMA} anos:                   {idade_acima_do_limite}")
print(f"Idade diverge do ano de nascimento (> 1 ano): {idade_divergente}")
print(f"Maior idade declarada:                       {idade_declarada.max():.0f} anos")

### 3.8 Possíveis outliers: atraso entre sintomas e notificação

O atraso (em dias) entre o início dos sintomas e a notificação indica a agilidade da vigilância. Valores muito altos podem ser erro de digitação na data ou uma notificação feita de fato muito depois.

In [ ]:
atraso_bruto = (datas["DT_NOTIFIC"] - datas["DT_SIN_PRI"]).dt.days
estatisticas_atraso = atraso_bruto.describe(percentiles=[0.5, 0.9, 0.99]).to_frame("dias").T
display(estatisticas_atraso.style.format("{:,.1f}"))

atraso_extremo = int((atraso_bruto > ATRASO_EXTREMO_DIAS).sum())
print(f"Notificações com atraso acima de {ATRASO_EXTREMO_DIAS} dias: {milhar(atraso_extremo)} ({percentual(atraso_extremo, len(df_original))})")
print("⚠ Um valor extremo, por si só, não prova erro: notificação tardia é possível.")

### 3.9 Diagnóstico consolidado

Todas as verificações num só lugar, inclusive as que **não** encontraram problema: elas também são evidência de que a base foi testada.

In [ ]:
total = len(df_original)
diagnostico = pd.DataFrame(
    [
        ("Q1", "Colunas 100% vazias", len(vazias), "colunas"),
        ("Q2", "Colunas com um único valor (TP_NOT, ID_AGRAVO, NU_ANO)", len(constantes), "colunas"),
        ("Q3", "Linhas duplicadas por completo", duplicatas_exatas, "registros"),
        ("Q4", "Possíveis duplicatas (mesma chave, linha diferente)", possiveis_duplicatas, "registros"),
        ("Q5", "Datas fora do formato AAAA-MM-DD", datas_formato_invalido, "valores"),
        ("Q6", "Registros com alguma data fora do período", registros_com_data_fora, "registros"),
        ("Q7", "Início dos sintomas depois da notificação", sintoma_depois_notificacao, "registros"),
        ("Q8", "Digitação antes da notificação", digitacao_antes_notificacao, "registros"),
        ("Q9", "Encerramento antes da notificação", encerramento_antes_notificacao, "registros"),
        ("Q10", f"Atraso sintomas → notificação acima de {ATRASO_EXTREMO_DIAS} dias", atraso_extremo, "registros"),
        ("Q11", "CLASSI_FIN fora do domínio (código '0')", classificacao_fora_dominio, "registros"),
        ("Q12", "CLASSI_FIN vazio (caso em investigação)", classificacao_vazia, "registros"),
        ("Q13", "Idade vazia ou com unidade inválida", idade_invalida, "registros"),
        ("Q14", f"Idade acima de {IDADE_MAXIMA} anos", idade_acima_do_limite, "registros"),
        ("Q15", "Idade diverge do ano de nascimento", idade_divergente, "registros"),
        ("Q16", "Sexo ignorado ou fora de M/F", sexo_sem_informacao, "registros"),
        ("Q17", "Diabetes sem sim/não (vazio ou 9)", diabetes_sem_informacao, "registros"),
        ("Q18", "Hipertensão sem sim/não (vazio ou 9)", hipertensao_sem_informacao, "registros"),
        ("Q19", "Hospitalizado sem data de internação", hospitalizado_sem_data, "registros"),
        ("Q20", "UF de residência vazia ou inexistente", uf_residencia_invalida, "registros"),
    ],
    columns=["ID", "verificação", "encontrados", "unidade"],
)
diagnostico["% da base"] = [
    percentual(quantidade, total) if unidade == "registros" else "-"
    for quantidade, unidade in zip(diagnostico["encontrados"], diagnostico["unidade"])
]
diagnostico["situação"] = np.where(diagnostico["encontrados"] > 0, "⚠ problema", "✓ sem problema")
display(diagnostico.style.format({"encontrados": "{:,.0f}"}).hide(axis="index"))

**Resposta do grupo:**

[Registre brevemente as principais características observadas.]

## 4. Limpeza e preparação dos dados

Realize a limpeza necessária ao projeto. Verifique valores ausentes, duplicados e inconsistências e trate os problemas encontrados, quando necessário.

### 4.1 Decidir

Limpeza não é só código: antes de transformar, registramos o que será feito com cada problema e por quê. O critério é **não apagar informação sem necessidade**: um registro com um campo ruim ainda serve para as perguntas que não dependem daquele campo.

In [ ]:
decisoes = pd.DataFrame(
    [
        ("D1", "Q3", "Duplicatas exatas", "Remover, mantendo uma ocorrência", "Linha idêntica nas 121 colunas é o mesmo registro repetido; mantê-la inflaria a contagem de casos (P1)."),
        ("D2", "Q4", "Possíveis duplicatas", "Manter e sinalizar em uma coluna", "Sem identificador de paciente, remover poderia apagar pessoas diferentes."),
        ("D3", "Q1, Q2, Q8, Q9, Q19", "121 colunas, a maioria fora das perguntas", "Manter só as colunas usadas por P1 a P3", "Colunas constantes não informam nada; problemas em colunas descartadas (digitação, encerramento, internação) ficam registrados, sem tratamento."),
        ("D4", "Q5, Q6, Q7", "Datas lidas como texto", "Converter para data; data de sintomas impossível vira vazia e sinalizada", "Datas precisam de tipo data para ordenar e agrupar por mês (P1)."),
        ("D5", "Q11, Q12", "CLASSI_FIN com '0' (só no ES) e vazio", "Agrupar como 'Sem classificação'", "'0' não existe no dicionário; o ES o usa no lugar do vazio. Esses casos ficam fora do denominador de gravidade."),
        ("D6", "Q13, Q14, Q15", "Idade codificada", "Converter para anos; acima de 120 vira vazio; usar o ano de nascimento como alternativa", "O código mistura unidade e valor; idade acima de 120 anos é impossível."),
        ("D7", "Q16", "Sexo 'I' (ignorado)", "Tornar vazio", "P2 compara homens e mulheres; 'ignorado' não informa."),
        ("D8", "Q17, Q18", "Comorbidades e hospitalização com 9 ou vazio", "Converter 1/2 para verdadeiro/falso e o resto para vazio", "A linha é mantida; a falta de informação só a exclui da análise daquela comorbidade (P3)."),
        ("D9", "Q20", "UF e município como códigos", "Traduzir para sigla e região, mantendo os códigos de município", "Siglas e regiões tornam a análise legível; nenhuma UF ficou sem tradução."),
        ("D10", "Q10", "Atrasos extremos", "Manter", "Notificação tardia é plausível; valor extremo, por si só, não justifica exclusão."),
    ],
    columns=["decisão", "diagnóstico", "problema", "tratamento", "justificativa"],
)
display(decisoes.style.hide(axis="index"))

### 4.2 Transformar

Cada tratamento é aplicado em `df_limpo` e mostra o estado antes, o resultado e o impacto.

In [ ]:
df_limpo = df_original.copy()

linhas_antes = len(df_limpo)
df_limpo = df_limpo.drop_duplicates()
df_limpo.insert(0, "id_registro", df_limpo.index)
df_limpo = df_limpo.reset_index(drop=True)

print("D1: DUPLICATAS EXATAS")
print("─" * 50)
print(f"Linhas antes:                {milhar(linhas_antes)}")
print(f"Duplicatas encontradas:      {milhar(duplicatas_exatas)}")
print(f"Linhas depois:               {milhar(len(df_limpo))}")
print(f"✓ Duplicatas restantes:      {df_limpo.drop(columns='id_registro').duplicated().sum()}")
print("id_registro guarda a posição da linha no arquivo original, para rastreabilidade.")

In [ ]:
df_limpo["possivel_duplicata"] = df_limpo.duplicated(CHAVE_POSSIVEL_DUPLICATA, keep=False).astype("boolean")

print("D2: POSSÍVEIS DUPLICATAS")
print("─" * 50)
print(f"Linhas sinalizadas:          {milhar(int(df_limpo['possivel_duplicata'].sum()))}")
print("A contagem inclui todas as linhas de cada grupo, inclusive a primeira.")
print("✓ Nenhuma linha removida; a coluna permite excluí-las numa análise de sensibilidade.")

In [ ]:
COLUNAS_MANTIDAS = ["id_registro", "possivel_duplicata"] + COLUNAS_DAS_PERGUNTAS
colunas_antes = df_limpo.shape[1]
df_limpo = df_limpo[COLUNAS_MANTIDAS]

print("D3: SELEÇÃO DE COLUNAS")
print("─" * 50)
print(f"Colunas antes:               {colunas_antes}")
print(f"Colunas depois:              {df_limpo.shape[1]}")
print(f"✓ Mantidas: {', '.join(COLUNAS_DAS_PERGUNTAS)}")

In [ ]:
tipo_antes = str(df_limpo["DT_SIN_PRI"].dtype)
notificacao = pd.to_datetime(df_limpo["DT_NOTIFIC"], format="%Y-%m-%d", errors="coerce")
sintomas = pd.to_datetime(df_limpo["DT_SIN_PRI"], format="%Y-%m-%d", errors="coerce")
sintomas_invalidos = df_limpo["DT_SIN_PRI"].notna() & (
    ~sintomas.between(INICIO_PERIODO, FIM_PERIODO) | (sintomas > notificacao)
)

df_limpo["dt_notificacao"] = notificacao
df_limpo["dt_sintomas"] = sintomas.mask(sintomas_invalidos)
df_limpo["dt_sintomas_invalida"] = sintomas_invalidos.astype("boolean")
df_limpo["semana_epi_sintomas"] = df_limpo["SEM_PRI"]
df_limpo = df_limpo.drop(columns=["DT_NOTIFIC", "DT_SIN_PRI", "SEM_PRI"])

print("D4: DATAS")
print("─" * 50)
print(f"Tipo antes:                  {tipo_antes}")
print(f"Tipo depois:                 {df_limpo['dt_sintomas'].dtype}")
print(f"Datas de sintomas inválidas: {int(sintomas_invalidos.sum())}")
print(f"Notificações sem data:       {int(df_limpo['dt_notificacao'].isna().sum())}")
print("✓ Datas convertidas; a regra de data impossível fica ativa para versões futuras do arquivo.")

In [ ]:
classificacao_antes = df_limpo["CLASSI_FIN"].fillna("<vazio>").value_counts().rename("antes")

codigo_valido = df_limpo["CLASSI_FIN"].where(df_limpo["CLASSI_FIN"].isin(list(CLASSIFICACAO_FINAL)))
df_limpo["classificacao_final"] = codigo_valido.map(CLASSIFICACAO_FINAL).fillna("Sem classificação")
df_limpo["criterio_confirmacao"] = df_limpo["CRITERIO"].map(CRITERIO_CONFIRMACAO)
df_limpo["evolucao"] = df_limpo["EVOLUCAO"].map(EVOLUCAO_INFORMADA)

classificacao_depois = df_limpo["classificacao_final"].value_counts().rename("depois")
df_limpo = df_limpo.drop(columns=["CLASSI_FIN", "CRITERIO", "EVOLUCAO"])

print("D5: CLASSIFICAÇÃO FINAL")
print("─" * 50)
print("Antes (códigos):")
display(classificacao_antes.to_frame().style.format("{:,.0f}"))
print("Depois (rótulos):")
display(classificacao_depois.to_frame().style.format("{:,.0f}"))
print(f"✓ '0' e vazio agrupados em 'Sem classificação': {milhar(int(classificacao_depois['Sem classificação']))}")
print("EVOLUCAO 9 (ignorado) e CRITERIO vazio passam a ser vazios.")

In [ ]:
codigo = df_limpo["NU_IDADE_N"].astype("string")
declarada = pd.to_numeric(codigo.str[1:], errors="coerce") * codigo.str[0].map(FATOR_IDADE_EM_ANOS).astype("float")
declarada = declarada.where(declarada.between(0, IDADE_MAXIMA))
pelo_nascimento = ANO_REFERENCIA - pd.to_numeric(df_limpo["ANO_NASC"], errors="coerce")
pelo_nascimento = pelo_nascimento.where(pelo_nascimento.between(0, IDADE_MAXIMA))

df_limpo["idade_anos"] = np.floor(declarada.fillna(pelo_nascimento)).astype("Int64")
recuperadas = int((declarada.isna() & pelo_nascimento.notna()).sum())
df_limpo = df_limpo.drop(columns=["NU_IDADE_N", "ANO_NASC"])

print("D6: IDADE")
print("─" * 50)
print(f"Idades inutilizáveis antes:  {int(declarada.isna().sum())}")
print(f"Recuperadas pelo nascimento: {recuperadas}")
print(f"Idades vazias depois:        {int(df_limpo['idade_anos'].isna().sum())}")
print(f"Intervalo final:             {df_limpo['idade_anos'].min()} a {df_limpo['idade_anos'].max()} anos")
print("✓ Exemplos: 4026 → 26 anos | 3008 → 0 anos (8 meses)")

In [ ]:
sexo_antes = df_limpo["CS_SEXO"].fillna("<vazio>").value_counts().rename("antes")
df_limpo["sexo"] = df_limpo["CS_SEXO"].map(SEXO_INFORMADO)
sexo_depois = df_limpo["sexo"].fillna("<vazio>").value_counts().rename("depois")
df_limpo = df_limpo.drop(columns=["CS_SEXO"])

print("D7: SEXO")
print("─" * 50)
display(pd.concat([sexo_antes, sexo_depois], axis=1).fillna(0).astype(int).style.format("{:,.0f}"))
print("✓ 'I' (ignorado) passou a vazio.")

In [ ]:
COLUNAS_SIM_NAO = {"HOSPITALIZ": "hospitalizado", "DIABETES": "diabetes", "HIPERTENSA": "hipertensao"}
resumo_sim_nao = []
for origem, destino in COLUNAS_SIM_NAO.items():
    df_limpo[destino] = df_limpo[origem].map(SIM_NAO).astype("boolean")
    resumo_sim_nao.append({
        "coluna": f"{origem} → {destino}",
        "sim": int(df_limpo[destino].eq(True).sum()),
        "não": int(df_limpo[destino].eq(False).sum()),
        "sem informação": int(df_limpo[destino].isna().sum()),
    })
df_limpo = df_limpo.drop(columns=list(COLUNAS_SIM_NAO))

print("D8: SIM / NÃO")
print("─" * 50)
display(pd.DataFrame(resumo_sim_nao).style.format({"sim": "{:,.0f}", "não": "{:,.0f}", "sem informação": "{:,.0f}"}).hide(axis="index"))
print("✓ 1 → verdadeiro, 2 → falso, 9 e vazio → sem informação.")

In [ ]:
def uf_e_regiao(codigo):
    uf = codigo.map(UF_POR_CODIGO)
    regiao = codigo.str[0].map(REGIAO_POR_PRIMEIRO_DIGITO).where(uf.notna())
    return uf, regiao

df_limpo["uf_notificacao"], df_limpo["regiao_notificacao"] = uf_e_regiao(df_limpo["SG_UF_NOT"])
df_limpo["uf_residencia"], df_limpo["regiao_residencia"] = uf_e_regiao(df_limpo["SG_UF"])
df_limpo["municipio_notificacao"] = df_limpo["ID_MUNICIP"]
df_limpo["municipio_residencia"] = df_limpo["ID_MN_RESI"]
df_limpo = df_limpo.drop(columns=["SG_UF_NOT", "SG_UF", "ID_MUNICIP", "ID_MN_RESI"])

print("D9: LOCALIZAÇÃO")
print("─" * 50)
print(f"UF de notificação sem tradução:  {int(df_limpo['uf_notificacao'].isna().sum())}")
print(f"UF de residência sem tradução:   {int(df_limpo['uf_residencia'].isna().sum())}")
print(f"UFs de residência distintas:     {df_limpo['uf_residencia'].nunique()}")
print("✓ Códigos traduzidos para sigla e região.")

**Resposta do grupo:**

[Indique os problemas encontrados e como foram tratados. Se não houve necessidade de tratamento, registre isso brevemente.]

## 5. Transformação dos dados

Realize as transformações necessárias à análise, como criação de colunas, filtros, agrupamentos, conversões ou junções, quando aplicáveis.

In [ ]:
FAIXAS_ETARIAS = [0, 5, 15, 30, 45, 60, 75, IDADE_MAXIMA + 1]
ROTULOS_FAIXAS = ["0-4", "5-14", "15-29", "30-44", "45-59", "60-74", "75+"]
GRAVIDADE = {"Dengue": False, "Dengue com sinais de alarme": True, "Dengue grave": True}

df_limpo["mes_sintomas"] = df_limpo["dt_sintomas"].dt.strftime("%Y-%m")
df_limpo["faixa_etaria"] = pd.cut(df_limpo["idade_anos"].astype("float"), bins=FAIXAS_ETARIAS, right=False, labels=ROTULOS_FAIXAS)
df_limpo["caso_grave"] = df_limpo["classificacao_final"].map(GRAVIDADE).astype("boolean")
df_limpo["atraso_notificacao_dias"] = (df_limpo["dt_notificacao"] - df_limpo["dt_sintomas"]).dt.days.astype("Int64")

print("COLUNAS DERIVADAS")
print("─" * 50)
print(f"mes_sintomas:             {df_limpo['mes_sintomas'].min()} a {df_limpo['mes_sintomas'].max()}")
print(f"faixa_etaria:             {', '.join(ROTULOS_FAIXAS)}")
print(f"caso_grave preenchido:    {milhar(int(df_limpo['caso_grave'].notna().sum()))} (classificações 10, 11 e 12)")
print(f"atraso acima de {ATRASO_EXTREMO_DIAS} dias:  {milhar(int((df_limpo['atraso_notificacao_dias'] > ATRASO_EXTREMO_DIAS).sum()))} (D10: mantidos)")

**Resposta do grupo:**

As transformações criam apenas as colunas que as perguntas exigem, sem agregar nem calcular indicadores (que ficam para os próximos RAs):

- `mes_sintomas` (AAAA-MM), a partir da data de início dos sintomas, para localizar o pico (P1);
- `faixa_etaria`, em sete faixas a partir da idade em anos, para comparar a gravidade entre idades (P3);
- `caso_grave`: verdadeiro para "dengue com sinais de alarme" e "dengue grave", falso para "dengue" e vazio para inconclusivos e casos sem classificação, que ficam fora do denominador de gravidade (P2 e P3);
- `atraso_notificacao_dias`: dias entre sintomas e notificação. Não é usado pelas perguntas do RA1, mas servirá ao RA2, que acompanha a chegada das notificações ao longo do tempo.

## 6. Conjunto de dados resultante

Validação das regras, gravação do arquivo preparado e comparação antes × depois.

### 6.1 Validar

Uma transformação não termina quando o comando executa. As regras abaixo verificam se o resultado corresponde às decisões; se alguma falhar, o `assert` interrompe o notebook antes de o arquivo ser gravado.

In [ ]:
idade = df_limpo["idade_anos"]
sintomas = df_limpo["dt_sintomas"]
com_gravidade = df_limpo["classificacao_final"].isin(list(GRAVIDADE))

validacoes = pd.DataFrame(
    [
        ("Linhas = base original menos as duplicatas exatas", len(df_limpo) == len(df_original) - duplicatas_exatas),
        ("id_registro é único", df_limpo["id_registro"].is_unique),
        ("Toda notificação tem data dentro do período", df_limpo["dt_notificacao"].between(INICIO_PERIODO, FIM_PERIODO).all()),
        ("Data de sintomas vazia ou dentro do período", (sintomas.isna() | sintomas.between(INICIO_PERIODO, FIM_PERIODO)).all()),
        ("Sintomas nunca depois da notificação", not (sintomas > df_limpo["dt_notificacao"]).any()),
        ("Atraso de notificação nunca negativo", not (df_limpo["atraso_notificacao_dias"] < 0).any()),
        (f"Idade vazia ou entre 0 e {IDADE_MAXIMA} anos", (idade.isna() | idade.between(0, IDADE_MAXIMA)).all()),
        ("Faixa etária vazia só quando a idade é vazia", (df_limpo["faixa_etaria"].isna() == idade.isna()).all()),
        ("Classificação final dentro do domínio", df_limpo["classificacao_final"].isin(list(CLASSIFICACAO_FINAL.values()) + ["Sem classificação"]).all()),
        ("caso_grave preenchido só nas classificações 10, 11 e 12", (df_limpo["caso_grave"].notna() == com_gravidade).all()),
        ("UF de residência é uma das 27 UFs", df_limpo["uf_residencia"].isin(list(UF_POR_CODIGO.values())).all()),
        ("Sexo vazio ou Masculino/Feminino", (df_limpo["sexo"].isna() | df_limpo["sexo"].isin(list(SEXO_INFORMADO.values()))).all()),
        ("df_original não foi alterado", df_original.shape == (linhas_no_arquivo, 121)),
    ],
    columns=["regra verificada", "resultado"],
)
validacoes["status"] = validacoes["resultado"].map({True: "✓ OK", False: "✗ REVISAR"})

print("VALIDAÇÃO FINAL")
print("─" * 50)
display(validacoes[["regra verificada", "status"]].style.hide(axis="index"))

falhas = validacoes.loc[~validacoes["resultado"], "regra verificada"].tolist()
assert not falhas, f"Validações que falharam: {falhas}"
print("✓ Todas as regras definidas foram atendidas.")

### 6.2 Organizar e salvar

O conjunto final é salvo em **Parquet**: o formato guarda os tipos (datas continuam datas, categorias continuam categorias), ocupa bem menos espaço que o CSV e é lido diretamente pelo pandas e pelo Spark, que será usado no RA2.

In [ ]:
COLUNAS_FINAIS = [
    "id_registro", "dt_notificacao", "dt_sintomas", "semana_epi_sintomas", "mes_sintomas",
    "uf_notificacao", "regiao_notificacao", "municipio_notificacao",
    "uf_residencia", "regiao_residencia", "municipio_residencia",
    "sexo", "idade_anos", "faixa_etaria",
    "classificacao_final", "caso_grave", "criterio_confirmacao", "evolucao", "hospitalizado",
    "diabetes", "hipertensao", "atraso_notificacao_dias", "dt_sintomas_invalida", "possivel_duplicata",
]
CATEGORIAS = [
    "uf_notificacao", "regiao_notificacao", "uf_residencia", "regiao_residencia",
    "sexo", "classificacao_final", "criterio_confirmacao", "evolucao",
]

df_preparado = df_limpo[COLUNAS_FINAIS].copy()
df_preparado[CATEGORIAS] = df_preparado[CATEGORIAS].astype("category")
df_preparado[["semana_epi_sintomas", "mes_sintomas", "municipio_notificacao", "municipio_residencia"]] = (
    df_preparado[["semana_epi_sintomas", "mes_sintomas", "municipio_notificacao", "municipio_residencia"]].astype("string")
)

ARQUIVO_PREPARADO.parent.mkdir(parents=True, exist_ok=True)
df_preparado.to_parquet(ARQUIVO_PREPARADO, index=False)

relido = pd.read_parquet(ARQUIVO_PREPARADO)
releitura_identica = relido.shape == df_preparado.shape and relido.dtypes.astype(str).equals(df_preparado.dtypes.astype(str))

print("CONJUNTO PREPARADO")
print("─" * 50)
print(f"Arquivo:            {ARQUIVO_PREPARADO.relative_to(RAIZ).as_posix()}")
print(f"Shape:              {milhar(df_preparado.shape[0])} linhas × {df_preparado.shape[1]} colunas")
print(f"Tamanho em disco:   {ARQUIVO_PREPARADO.stat().st_size / 1e6:.1f} MB (bruto: {ARQUIVO_BRUTO.stat().st_size / 1e6:.1f} MB compactado)")
print(f"Memória em uso:     {df_preparado.memory_usage(deep=True).sum() / 1e6:.0f} MB (bruto: {df_original.memory_usage(deep=True).sum() / 1e6:.0f} MB)")
assert releitura_identica, "O arquivo relido difere do conjunto preparado."
print("✓ Releitura do Parquet idêntica ao conjunto preparado.")
display(df_preparado.head())

### 6.3 Comparação antes e depois

In [ ]:
comparacao = pd.DataFrame(
    [
        ("Registros", milhar(len(df_original)), milhar(len(df_preparado)), "D1: remoção das duplicatas exatas"),
        ("Atributos", df_original.shape[1], df_preparado.shape[1], "D3: colunas das perguntas, traduzidas, mais colunas derivadas"),
        ("Duplicatas exatas", duplicatas_exatas, int(df_preparado["id_registro"].duplicated().sum()), "D1"),
        ("Datas guardadas como texto", len(COLUNAS_DATA), 0, "D4: conversão para data"),
        ("CLASSI_FIN fora do domínio", milhar(classificacao_fora_dominio), 0, "D5: 'Sem classificação'"),
        ("Idade inutilizável", idade_invalida + idade_acima_do_limite, int(df_preparado["idade_anos"].isna().sum()), "D6: recuperada pelo ano de nascimento quando possível"),
        ("Sexo sem M/F", milhar(sexo_sem_informacao), milhar(int(df_preparado["sexo"].isna().sum())), "D7: mantidos, com sexo vazio"),
        ("Diabetes sem sim/não", milhar(diabetes_sem_informacao), milhar(int(df_preparado["diabetes"].isna().sum())), "D8: mantidos, com valor vazio"),
        ("Hipertensão sem sim/não", milhar(hipertensao_sem_informacao), milhar(int(df_preparado["hipertensao"].isna().sum())), "D8: mantidos, com valor vazio"),
        (f"Atraso acima de {ATRASO_EXTREMO_DIAS} dias", milhar(atraso_extremo), milhar(int((df_preparado["atraso_notificacao_dias"] > ATRASO_EXTREMO_DIAS).sum())), "D10: mantidos"),
        ("Linhas em grupos de possível duplicata", "-", milhar(int(df_preparado["possivel_duplicata"].sum())), "D2: mantidas e sinalizadas"),
    ],
    columns=["medida", "após ingestão", "após preparação", "explicação"],
)
display(comparacao.style.hide(axis="index"))

### 6.4 Dicionário do conjunto preparado

In [ ]:
descricoes = {
    "id_registro": ("posição no arquivo bruto", "rastreabilidade até o DENGBR26"),
    "dt_notificacao": ("DT_NOTIFIC", "data da notificação"),
    "dt_sintomas": ("DT_SIN_PRI", "início dos sintomas; vazia quando impossível"),
    "semana_epi_sintomas": ("SEM_PRI", "semana epidemiológica (AAAASS)"),
    "mes_sintomas": ("derivada", "AAAA-MM dos sintomas"),
    "uf_notificacao": ("SG_UF_NOT", "sigla da UF de notificação"),
    "regiao_notificacao": ("SG_UF_NOT", "região da UF de notificação"),
    "municipio_notificacao": ("ID_MUNICIP", "código IBGE, 6 dígitos"),
    "uf_residencia": ("SG_UF", "sigla da UF de residência"),
    "regiao_residencia": ("SG_UF", "região da UF de residência"),
    "municipio_residencia": ("ID_MN_RESI", "código IBGE, 6 dígitos"),
    "sexo": ("CS_SEXO", "Masculino / Feminino; vazio quando ignorado"),
    "idade_anos": ("NU_IDADE_N, ANO_NASC", f"anos completos, 0 a {IDADE_MAXIMA}"),
    "faixa_etaria": ("derivada", ", ".join(ROTULOS_FAIXAS)),
    "classificacao_final": ("CLASSI_FIN", "rótulo do dicionário ou 'Sem classificação'"),
    "caso_grave": ("derivada", "sinais de alarme ou grave = verdadeiro; dengue = falso"),
    "criterio_confirmacao": ("CRITERIO", "laboratorial, clínico-epidemiológico, em investigação"),
    "evolucao": ("EVOLUCAO", "cura ou tipo de óbito; vazio quando ignorado"),
    "hospitalizado": ("HOSPITALIZ", "verdadeiro / falso / vazio"),
    "diabetes": ("DIABETES", "verdadeiro / falso / vazio"),
    "hipertensao": ("HIPERTENSA", "verdadeiro / falso / vazio"),
    "atraso_notificacao_dias": ("derivada", "dias entre sintomas e notificação"),
    "dt_sintomas_invalida": ("derivada", "verdadeiro quando DT_SIN_PRI foi descartada"),
    "possivel_duplicata": ("derivada", "verdadeiro quando outra linha tem a mesma chave"),
}
dicionario_final = pd.DataFrame(
    [(coluna, *descricoes[coluna], str(df_preparado[coluna].dtype), percentual(int(df_preparado[coluna].isna().sum()), len(df_preparado))) for coluna in COLUNAS_FINAIS],
    columns=["atributo", "origem", "descrição", "tipo", "ausente"],
)
display(dicionario_final.style.hide(axis="index"))

## 7. Considerações do RA1

Registre brevemente:

- O que o grupo descobriu sobre os dados.
- Quais dificuldades ou limitações encontrou.
- Como os dados preparados poderão ser utilizados nas próximas etapas.

**Resposta do grupo:**

[Preencha aqui.]